# Movie1 • Nov 22, 2026 incremental batch (21 photo posts)

This notebook handles **only source IDs 3835–3855** from `ChatExport_2026-11-22.zip`. It will not load or repost the earlier 3,780-message migration.

**One-time Kaggle setup:** Create a Kaggle dataset by uploading `ChatExport_2026-11-22.zip`, then attach it using **Add Input** to this notebook. Kaggle may automatically **extract ZIP uploads into a folder**; this fixed version detects both a ZIP file and the extracted `result.json` + photos folder automatically. Kaggle must have **Internet enabled** and the existing `TELEGRAM_BOT_TOKEN` secret ticked. For a background run use **Save Version → Save & Run All**. No Google Drive mount is needed.

Optional GitHub upload: To push every extracted asset in one structured commit to `kawakaze430/Movie1`, add a Kaggle secret called `GITHUB_TOKEN` containing a fine-grained GitHub token with **Contents: Read/Write** for that repository. Existing Kaggle/GitHub linkage does *not* automatically provide this token. If this secret is missing, publishing to GitHub is skipped and the Telegram step still works using the attached ZIP.

The notebook validates the source ZIP checksum, all 21 photos, all 61 preserved external links, and channel ID/title/admin status before sending. Telegram channel-promotion URLs are removed. A JSON checkpoint is saved after each confirmed post in `/kaggle/working`; attach that output to future executions to resume safely. **Never rerun from scratch after a successful or partially successful send unless the checkpoint is restored.**


In [ ]:
#@title 1. Configuration (no edits required)
import os, json, pathlib, hashlib, zipfile, re, io, time, sys, shutil, mimetypes
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

BATCH_NAME = 'ChatExport_2026-11-22'
EXPECTED_IDS = list(range(3835,3856))
EXPECTED_SOURCE_ID = 1881963160
DESTINATION_ID = -1004492918137
DESTINATION_TITLE = 'Bollywood Movie'
ZIP_SHA256 = 'dae307085f6a9ba3a7b52a14575640112811dbf793fea1391107d82bfeb02d52'
RESULT_SHA256 = '160cb4832abb4a12bcd678ee55a4458638c3acdeb5f1eb12c3aa504440082725'
EXPECTED_LINKS = 61
EXPECTED_PHOTOS = {'3835': '0c4dd9eeafffd647f26a0e563e76a45d3384033abd5c95adce9189cafdd226be', '3836': '7ccb70fab69bff3af36dd79a38da281ec91b4bde88983618d1d8ff8b8bf74bba', '3837': '23b0fae647b17c3c60c606667cd7b088a770b1cc2166e28c9c3b9f4bc7775031', '3838': 'be3fed728049e2c81773a98ae6d199b908916b9f807ace8d6b8731859b8e4969', '3839': '4692006777bcd5f8073a76c682ca8a7001b8f977a76ae6a414b170f4ac60944e', '3840': 'ec00b1f555724dc1a5a94139e18d6747d99d66039d48c07eacc9471bc1ff2d29', '3841': '4689082e2a0ad9f93666fecd5959f5ad597c06c0beba63bdceafafc61c0c4173', '3842': '691330080dfe7141fe0d2d1d49d88a18b38f45f05462f4873d92bb997fddf35f', '3843': 'cb60ed8e3e323355fd940ccd913a0f0727a9554cec1bb0ca85baab8578b4ce0a', '3844': 'b1ca10b24ad2f88c0020fdb99c2c626f6cde967826442cd26781dd3f9e147a1d', '3845': 'ac7036a5e44c468c459076ca990329219439145e0cc63e6dda0d72320ed8ad50', '3846': '126f9117d30491735d098b408b85bf4f9c41c69dca44fced09f3fef9aeedfa19', '3847': '169956d9e87c203e65f3086297c39c0a7de8235b017ddc7600bc70050a0bbadd', '3848': '389624f1bbbfce5e4e7aeccb867dcf034f37ad555bb92035f814ca66afd966fa', '3849': 'f1f3e5f4f51aa8101f31a08bc518df1c0a85d341522ce200d7f660ee3c2a7ec2', '3850': 'fc5cfc0a0a76647d123619708837f0d480be31bf6916401c5b5ecac81cd7c21b', '3851': '573b0585e05454edceb379f93ab769402cb1c04da3452d985b42550c784ab19e', '3852': '74189dad1931cde912a47ed7bd33a8c851cb933a1c35f8de8ec06503d31f4c0c', '3853': 'e4036ff5f0cfa1744abda0e3ea12c9ddfaa340a139d0b6a463c82b44878977d1', '3854': '69e3cffdd8076637100aaf348475f1805c92b3c7bcf451b6fdaa209f08ff0edb', '3855': '3df2cb35d07ff0d230dfebe13f9d12ab2d0f1f1a9a1207e2c0d62a5693348708'}
REPO_FULL_NAME = 'kawakaze430/Movie1'
REPO_BRANCH = 'main'
REPO_PATH = 'telegram_exports/2026-11-22'
RUN_MODE = 'SEND_21'  # Change to PREVIEW to check captions without sending.
SYNC_REPO_IF_GITHUB_SECRET_AVAILABLE = True
DELAY_SECONDS = 3.0
CHECKPOINT_NAME = f'progress_movie1_nov22_{DESTINATION_ID}.json'
WORK = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle').exists() else pathlib.Path('/mnt/data/nov22_test')
WORK.mkdir(parents=True, exist_ok=True)
LOCAL = WORK / 'movie1_nov22_export'
LOCAL.mkdir(parents=True, exist_ok=True)
print('Batch:', BATCH_NAME, '| expected posts:',len(EXPECTED_IDS),'| destination:',DESTINATION_ID)

def from_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(name) or '').strip()
    except Exception:
        return ''

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def safe_extract(z, prefix, dest):
    count=0
    for item in z.infolist():
        if item.is_dir() or not item.filename.startswith(prefix): continue
        rel = item.filename[len(prefix):]
        if not rel or rel.startswith('/') or '..' in pathlib.PurePosixPath(rel).parts: raise ValueError('Unsafe ZIP entry: '+rel)
        output = dest / rel
        output.parent.mkdir(parents=True,exist_ok=True)
        with z.open(item) as f, output.open('wb') as out: shutil.copyfileobj(f,out)
        count += 1
    return count

# Kaggle may unzip an uploaded archive automatically into nested input folders.
# Accept either the ZIP or an already-extracted export, but only if the
# result.json checksum matches this exact 21-message batch.
SOURCE_SEARCH_ROOTS = (
    pathlib.Path('/kaggle/input'),
    pathlib.Path('/kaggle/working'),
    pathlib.Path('/mnt/data'),
)

def discover_source():
    print('Finding new 21-post export in Kaggle input (ZIP or extracted folder)...')
    # First, look for an already-extracted Kaggle input, regardless of the
    # dataset slug or number of nested parent folders.
    for root in SOURCE_SEARCH_ROOTS:
        if not root.is_dir():
            continue
        for result_file in root.rglob('result.json'):
            if not result_file.is_file() or sha256(result_file) != RESULT_SHA256:
                continue
            source_dir = result_file.parent
            data = json.loads(result_file.read_text(encoding='utf-8'))
            photo_paths = [m.get('photo') for m in data.get('messages', []) if m.get('photo')]
            if not photo_paths or not all((source_dir / rel).is_file() for rel in photo_paths):
                print('Matching result.json found but some referenced photos are absent:', source_dir)
                continue
            print('Input: extracted folder verified:', source_dir)
            return source_dir

    # Otherwise look for the original uploaded ZIP and safely extract it.
    candidates = []
    for root in SOURCE_SEARCH_ROOTS:
        if root.is_dir():
            candidates.extend(root.rglob('ChatExport_2026-11-22.zip'))
            candidates.extend(root.rglob('Movie1_Nov22_Structured_21_Posts.zip'))
    for path in candidates:
        if path.name == 'ChatExport_2026-11-22.zip' and sha256(path) == ZIP_SHA256:
            with zipfile.ZipFile(path) as z:
                safe_extract(z, 'ChatExport_2026-11-22/', LOCAL)
            print('Input: original ZIP verified:', path)
            return LOCAL
    for path in candidates:
        if path.name == 'Movie1_Nov22_Structured_21_Posts.zip':
            with zipfile.ZipFile(path) as z:
                safe_extract(z, 'raw/', LOCAL)
            if (LOCAL / 'result.json').is_file() and sha256(LOCAL / 'result.json') == RESULT_SHA256:
                print('Input: structured ZIP verified:', path)
                return LOCAL

    # Optional GitHub fallback works only after the full raw media is uploaded.
    import urllib.parse
    rawbase = f'https://raw.githubusercontent.com/{REPO_FULL_NAME}/{REPO_BRANCH}/{REPO_PATH}/raw/'
    res = requests.get(rawbase + 'result.json', timeout=30)
    if res.status_code == 200 and hashlib.sha256(res.content).hexdigest() == RESULT_SHA256:
        (LOCAL / 'result.json').write_bytes(res.content)
        msgs = json.loads(res.content)['messages']
        for message in msgs:
            rel = message.get('photo')
            if rel:
                url = rawbase + urllib.parse.quote(rel, safe='/')
                r = requests.get(url, timeout=90)
                r.raise_for_status()
                f = LOCAL / rel
                f.parent.mkdir(parents=True, exist_ok=True)
                f.write_bytes(r.content)
        print('Input: GitHub extracted raw assets verified')
        return LOCAL
    raise RuntimeError(
        'SOURCE DATA NOT FOUND. Attach the original ChatExport_2026-11-22.zip '
        'as a Kaggle Input, or attach the Kaggle extracted dataset folder '
        'that contains result.json and photos/. ZERO Telegram posts sent.'
    )

SOURCE = discover_source()
if sha256(SOURCE/'result.json') != RESULT_SHA256: raise RuntimeError('Wrong source result.json (checksum mismatch).')


In [ ]:
#@title 2. Prepare and validate exactly these 21 posts (no Telegram sends)
def flatten(entities):
    if isinstance(entities, str): return entities
    if isinstance(entities, list): return ''.join(flatten(p) for p in entities)
    if isinstance(entities, dict): return flatten(entities.get('text',''))
    return ''

def clean_caption(text):
    text = re.split(r'\n\s*(?:All private channels|🔞\s*All adult channel)',text,maxsplit=1,flags=re.I)[0]
    text = re.sub(r'(?i)https?://(?:t\.me|telegram\.me|telegram\.dog)/[^\s]+', '', text)
    text = re.sub(r'(?m)^\s*[^\n]*?(?:All private channels|All adult channel)[^\n]*$', '', text, flags=re.I)
    text = text.rstrip()
    text = re.sub(r'\n[ \t]*[▬━─]{3,}[ \t]*$', '', text).rstrip()
    return text

result = json.loads((SOURCE/'result.json').read_text(encoding='utf-8'))
if result.get('id') != EXPECTED_SOURCE_ID: raise RuntimeError('Unexpected source channel')
messages = sorted(result.get('messages',[]),key=lambda m:m['id'])
if [m['id'] for m in messages] != EXPECTED_IDS: raise RuntimeError('Wrong exported range. Expected exactly IDs 3835..3855')
POSTS=[]
for m in messages:
    photo = m['photo']
    path = SOURCE / photo
    if not path.is_file(): raise RuntimeError('Missing original photo: '+photo)
    if sha256(path) != EXPECTED_PHOTOS[str(m['id'])]: raise RuntimeError('Photo checksum mismatch: '+photo)
    orig=flatten(m.get('text',[]))
    cleaned=clean_caption(orig)
    external=lambda t: re.findall(r'https?://(?!t\.me|telegram\.me)[^\s]+',t,flags=re.I)
    if external(cleaned)!=external(orig): raise RuntimeError('An original non-channel URL was lost: '+str(m['id']))
    if 't.me/' in cleaned or 'All private channels' in cleaned: raise RuntimeError('Channel advertisement remained: '+str(m['id']))
    if len(cleaned)>1024: raise RuntimeError('Photo caption too long: '+str(m['id']))
    if not cleaned.strip() or 'Genre:' not in cleaned and 'Genre:' not in cleaned.replace('📼',''): raise RuntimeError('Unexpected caption text: '+str(m['id']))
    POSTS.append({'source_id':m['id'], 'title':cleaned.splitlines()[0].strip(), 'photo':photo, 'photo_sha256':EXPECTED_PHOTOS[str(m['id'])], 'caption':cleaned, 'non_channel_links':external(cleaned)})
if len(POSTS)!=21 or sum(len(p['non_channel_links']) for p in POSTS)!=EXPECTED_LINKS: raise RuntimeError('Incomplete batch')
print('READY: 21/21 photos validated | all 61 non-channel links retained | no Telegram channel links')
for p in POSTS: print(p['source_id'],'|',p['title'],'| links:',len(p['non_channel_links']))
print('NO POSTS SENT during preparation.')


In [ ]:
#@title 3. Optional: commit all extracted original files + cleaned data to GitHub
# This cell does not require GitHub access to post to Telegram. It only runs if the GITHUB_TOKEN secret exists.
GH_TOKEN = from_secret('GITHUB_TOKEN') if SYNC_REPO_IF_GITHUB_SECRET_AVAILABLE else ''

def github_put_archive(source_dir, posts, token):
    # REST Git Data API: one commit, instead of 90 individual commits or cloning a 500MB repo.
    base_url='https://api.github.com/repos/'+REPO_FULL_NAME
    session=requests.Session()
    session.headers.update({'Authorization':'Bearer '+token,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
    def api(method,path,body=None):
        r=session.request(method,base_url+path,json=body,timeout=120)
        if r.status_code>=400: raise RuntimeError(f'GitHub {method} {path} HTTP {r.status_code}: {r.text[:240]}')
        return r.json()
    ref=api('GET','/git/ref/heads/'+REPO_BRANCH)
    parent=ref['object']['sha']
    commit=api('GET','/git/commits/'+parent)
    original_files = [f for f in source_dir.rglob('*') if f.is_file()]
    # We keep every one of the Telegram export viewer assets, not just photo posters.
    repo_files={REPO_PATH+'/raw/'+str(p.relative_to(source_dir)).replace(chr(92),'/'):p for p in original_files}
    manifest={
       'batch':BATCH_NAME,'source_ids':[3835,3855],'message_count':21,
       'photo_count':21,'download_links_preserved':EXPECTED_LINKS,
       'source_zip_sha256':ZIP_SHA256, 'source_result_sha256':RESULT_SHA256,
       'target_chat_id':DESTINATION_ID,
    }
    extra={
       REPO_PATH+'/processed/posts.jsonl':''.join(json.dumps(p,ensure_ascii=False)+'\n' for p in posts).encode('utf-8'),
       REPO_PATH+'/processed/manifest.json':(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n').encode('utf-8'),
       REPO_PATH+'/README.md':('Incremental Movie1 export (2026-11-22): 21 source posts 3835-3855. The raw folder contains the original Telegram export (photo media + HTML viewer). The processed folder preserves all 61 non-channel download links and removes Telegram promotional links.\n').encode('utf-8')
    }
    todo = [(name,path.read_bytes()) for name,path in repo_files.items()]
    todo.extend(extra.items())
    import base64
    def upload_blob(item):
        path, content=item
        payload={'content':base64.b64encode(content).decode('ascii'), 'encoding':'base64'}
        r=requests.post(base_url+'/git/blobs',json=payload,headers=session.headers,timeout=120)
        if r.status_code>=400: raise RuntimeError(f'Blob upload {path} HTTP {r.status_code}: {r.text[:240]}')
        return {'path':path,'mode':'100644','type':'blob','sha':r.json()['sha']}
    with ThreadPoolExecutor(max_workers=5) as pool:
        entries=list(pool.map(upload_blob,todo))
    newtree=api('POST','/git/trees',{'base_tree':commit['tree']['sha'],'tree':entries})
    newcommit=api('POST','/git/commits',{'message':f'Add Movie1 incremental batch 2026-11-22 (21 posts)', 'tree':newtree['sha'],'parents':[parent]})
    api('PATCH','/git/refs/heads/'+REPO_BRANCH,{'sha':newcommit['sha'],'force':False})
    return newcommit['sha'],len(todo)

if GH_TOKEN:
    print('GitHub token found; syncing the structured export to',REPO_FULL_NAME)
    try:
        sha, count=github_put_archive(SOURCE,POSTS,GH_TOKEN)
        print('GitHub sync complete:',count,'files in ONE commit:',sha)
        print('https://github.com/'+REPO_FULL_NAME+'/tree/'+sha+'/'+REPO_PATH)
    except Exception as exc:
        print('WARNING: GitHub sync did not complete. No repository completion claimed:',type(exc).__name__,str(exc)[:350])
else:
    print('GitHub sync skipped (GITHUB_TOKEN Kaggle secret not configured). Telegram upload can still proceed using the attached ZIP.')


In [ ]:
#@title 4. Post ONLY source IDs 3835–3855 to Bollywood Movie, with checkpoint
# Running this step sends real Telegram messages; use RUN_MODE=PREVIEW in step 1 to disable.
CHECKPOINT=WORK/CHECKPOINT_NAME
previous=None
if not CHECKPOINT.exists():
    for d in [pathlib.Path('/kaggle/input'),pathlib.Path('/mnt/data')]:
        if d.exists():
            for p in d.rglob(CHECKPOINT_NAME):
                if p!=CHECKPOINT:
                    previous=p
                    shutil.copyfile(p,CHECKPOINT)
                    break
        if previous: break

def save_state(s):
    temp=CHECKPOINT.with_suffix('.json.tmp')
    with open(temp,'w',encoding='utf-8') as f:
        json.dump(s,f,ensure_ascii=False,indent=2)
        f.flush(); os.fsync(f.fileno())
    os.replace(temp,CHECKPOINT)

if RUN_MODE!='SEND_21':
    print('PREVIEW ONLY. No messages sent.')
else:
    bot_token=from_secret('TELEGRAM_BOT_TOKEN')
    if not bot_token: raise RuntimeError('Enable your TELEGRAM_BOT_TOKEN Kaggle secret. ZERO posts sent.')
    base_api='https://api.telegram.org/bot'+bot_token+'/'
    def telegram_call(method,data=None,files=None,timeout=30):
        r=requests.post(base_api+method,data=data,files=files,timeout=(15,timeout))
        try: payload=r.json()
        except ValueError: raise RuntimeError('Telegram returned non-JSON (delivery status may be uncertain)')
        if not payload.get('ok'):
            return {'error':payload, 'status_code':r.status_code}
        return {'result':payload['result']}
    me=telegram_call('getMe')
    if 'error' in me: raise RuntimeError('Bot token rejected. ZERO posts sent.')
    bot_id=me['result']['id']
    chat=telegram_call('getChat',{'chat_id':DESTINATION_ID})
    if 'error' in chat or chat['result']['id']!=DESTINATION_ID or chat['result'].get('title')!=DESTINATION_TITLE:
        raise RuntimeError('Wrong destination channel. ZERO posts sent.')
    member=telegram_call('getChatMember',{'chat_id':DESTINATION_ID,'user_id':bot_id})
    if 'error' in member or member['result'].get('status') not in ('administrator','creator') or member['result'].get('can_post_messages') is False:
        raise RuntimeError('Bot must be a channel admin with posting rights. ZERO posts sent.')
    print('DESTINATION VERIFIED:',DESTINATION_TITLE,'(',DESTINATION_ID,') | bot:',me['result'].get('username'))
    if CHECKPOINT.exists():
        state=json.loads(CHECKPOINT.read_text(encoding='utf-8'))
        if state.get('batch')!=BATCH_NAME or state.get('destination_id')!=DESTINATION_ID or state.get('bot_id')!=bot_id or state.get('source_sha256')!=RESULT_SHA256:
            raise RuntimeError('Existing checkpoint does not match this batch, bot, or target. ZERO new posts sent.')
    else:
        state={'batch':BATCH_NAME,'destination_id':DESTINATION_ID,'bot_id':bot_id,'source_sha256':RESULT_SHA256,'sent':{},'pending':None}
        save_state(state)
        print('Starting NEW 21-post batch. If these 21 posts were already uploaded in another Kaggle run, stop now and restore its checkpoint.')
    if state.get('pending'):
        raise RuntimeError('A prior send is pending/uncertain. Inspect latest Telegram post before resolving it manually; no automatic retry: '+str(state['pending']))
    print('Previously confirmed:',len(state['sent']),'of 21 | checkpoint:',CHECKPOINT)
    for index,p in enumerate(POSTS,1):
        key=str(p['source_id'])
        if key in state['sent']: continue
        state['pending']={'source_id':p['source_id'],'title':p['title']}
        save_state(state)
        while True:
            try:
                with open(SOURCE/p['photo'],'rb') as photo_file:
                    r=telegram_call('sendPhoto',data={'chat_id':DESTINATION_ID,'caption':p['caption']},files={'photo':(pathlib.Path(p['photo']).name,photo_file,'image/jpeg')},timeout=140)
            except Exception as exc:
                print('STOP: network exception during send; outcome uncertain:',type(exc).__name__,str(exc)[:250]); raise
            if 'result' in r:
                message_id=r['result'].get('message_id')
                if not message_id: raise RuntimeError('Telegram response has no message ID; outcome uncertain. Stopping.')
                state['sent'][key]=message_id
                state['pending']=None
                save_state(state)
                print(f'SENT {len(state["sent"])}/21 | source {p["source_id"]} | Telegram message {message_id} | {p["title"]}',flush=True)
                break
            err=r['error']
            if err.get('error_code')==429:
                wait=int(err.get('parameters',{}).get('retry_after',15)) + 1
                print('Telegram rate limit: waiting',wait,'seconds')
                time.sleep(wait)
                continue
            raise RuntimeError('Telegram did not accept this post; pending retained: '+str(err)[:400])
        time.sleep(DELAY_SECONDS)
    print('DONE: confirmed 21/21 new photo posts sent to Bollywood Movie!')
    print('Save and preserve checkpoint:',CHECKPOINT)
